### Runtime vs compile-time

**Recap**

in the initializer of a compile-time var can appear:  
1- literals  
2- other compile-time vars  
3- return values of 'constexpr' funcs, called with args that are compile-time const

**What to call vars**

each time i define a var:  

1- that is a compile-time constant: call it 'constexpr', not 'const'  
this way, it will be easier later, when declaring new vars, to see if the initializer is a const expression or not  

2- that is a runtime const: call it 'const'  

3- of a type that cannot be 'constexpr', like string or vector: either call it 'const', or pick a different type that is constexpr compatible (string_view, array)

**Runtime vs compile-time constants**

an object must be one of these, depending on when its value is known  
some examples are:    

1- compile-time constant    
Literals (any type)  
Const objects, of INTEGRAL TYPE, whose initializers are compile-time constants  
constexpr objects (any type)  
return value of a constexpr function, which was called with args that are compile-time constants     

2- run-time constant  
Constant function parameters  
Constant objects, whose initializers are NOT compile-time constants (so initializers are non-constants or runtime constants)       
const objects, NOT of integral type  
return value of a non-constexpr function (even if technically could know it at compile-time, bc if func is not constexpr it don't even try to) 

**Attention: floating point vars** 

to be compile-time constants:  

1- INTEGRAL vars:   
initializers must be compile-time expressions   
but when declaing the var, 'const' is enough  

2- FLOATING POINT vars:    
initializers must be compile-time expressions   
and when declaing the var, MUST be 'constexpr'    

example:  
const int i { 1 };          // compile-time  
const double f { 1.2 };     // runtime  
constexpr double g { 1.2 }; // compile-time  

**Example**

code | what is it
---|---
int a { 5 }; | non-constant  
const int b { 5 }; | compile-time constant <br> bc its initializer is a literal, so a compile-time constant  
const int d { b }; | compile-time constant <br> bc its initializer, b, is a compile-time const
const int e { a }; | runtime constant <br> bc initializer is a non-const
const int f { e } | runtime constant <br> bc its initializer is a runtime const, not a compile-time const
int five() <br> { return 5; } <br> const int g { five() }; | runtime constant <br> bc ret value isn't known until runtime, since it's not a constexpr func  
int pass(const int x) <br> { return x; } <br>const int h { pass(5) };  | runtime constant <br> bc ret value isn't known until runtime, since it's not a constexpr func    
constexpr int f(a,b);  <br> const c { f(1,3) }; | compile-time constant <br> bc initializer contains the ret value of constexpr func, anf the args with which it's called are compile-time constants  
const double f { 1.2 }; | runtime constant <br> bc it's non-integrap type, so const is not enough, even if intializer is compile-time const    
constexpr double g { 1.2 }; | compile-time <br> bc all stuff with 'constexp' is compile-time constant, no matter the type

#### Notes

**Compile-time**

just reading the code, without running it  
so, not passing it any input, nor doing any dynamic allocation  

**Why cannot string be compile-time constants??? CHECK**

can mark an object constexpr only if the compiler can build that object completely during compilation, without requiring any runtime storage, code execution, or dynamic initialization  

building a std::string and std::vector, require dynamic memory allocation, which is a run-time construct   
so they cannot be constexpr  

**Function parameters**

int f(const int x)
 
1- can be const, means they are runtime constants    
means they do not change value during the function  

2- CANNOT be constexpr  
since their valus must be evaluated at runtime (bc the func could be called more times, with different args)  

### Const vs constexpr

#### Applied to a var

what | const | constexpr  
---|---|---
with what var types can be used | any kind of type | yes: integrals, reference, array <br> no: string, vector  
meaning | 1- read-only | 1- read-only <br> 2- compile-time value required  
how to write | `const int x` | `constexpr int x`

**Meaning: explanation**

1- read-only  
cannot modify value after initialization    
(so, in different runs can be initialized at different values)  

2- compile-time value required  
the initializer (expression on the right of the =) must be evaluatable at compile time (be a compile-time expression, so contain only compile-time constants)    
does NOT mean that the var is initialzed at run-time: just means that the value is evaluated, but storage for the object does not exist yet  

**Relationship**

constexpr is a stricter version of const  (adds requirement 2, of compile-time value)    
so, has no sense putting both in same var   

difference:  
in different runs, wihtout changing the code:  
const: can be initialized to different values, bc depends on some value i'll get at run-time, maybe an input  
constexpr: is always initialized at same value, bc the value is entirely determined by the code  

**Function params**

sometimes params are given type const   
means the function must not modify an argument passed by reference or pointer (by value makes no sense saying it, it makes a copy anyway, so it couldn't change the original value even if wanted to)   

int f(int const)

#### Applied to a func

**Definitions**

what | const | constexpr  
---|---|---
where can be used | only in member funcs (funcs of a class) <br> and those funcs can only be called by const objects | in any func  
meaning | does not modify any property of the object | compile-time-capable (if passed suitable args)
how to write it | `int f() const {}`  | `constexpr int f() {}`   

note: writing 'const' before 'int' mean the function returns a const int by value  
this is usually unnecessary  

**Relationship**

mean different things, so can both be applied to a same func (only to a meber func, be const can only be applied there)  
in this case, means the func is both:  
1- compile-time-capable   
2- does not modify the object

**Constexpr: does it run at compile-time?**

MIGHT be evaluated at compile time:  
1- if called with args evaluatable at compile-time: MUST be evaluted at compile time  
2- if called with args evaluatable at runtime: doesn't matter, it's evaluted at runtime

essentially, IF ir can be done at compile-time, it does it      
but it's not required, can also use it normally   

This capability of evaluation of constexpr function in both compile-time and run-time based on the context its being used is a great advantage  
So programmers does not have to write two separate overloaded function with different name but completely duplicate code  

**Mutable**

can only be used in a class parameter  
means that parameter CAN be modified, even inside a constant function  

### Const

**Define**

const is a QUALIFIER  
it can be either before or after the type, but conventionally put before  

MUST be initialized when declared, not later  
there is not a convention on how to name them (like normal vars, or all uppercase)

**Best pratice**

if a var can be made const, do it  
const vars are operated on faster, so less runtime  

**In funcs**

1- conat parameters    
int f(int const x) {}  
means that the value/arg received cannot change inside the func  

2- return type const  
const int f(int x)

do not do either, they're bad things??? not even the first one???

### Const vs macro

**Types**

named constants: const vars  
1- const var  
2- macro with substitution text (preprocessor def)  
3- enumerated constant

literal constants: literals? what is the difference???

**Const var vs preprocessore macros**

const var are better  

1- bc macros replace each time that word appears, even if it's part of a longer word, or it's something else  
so give errors  

2- macros are harder to debug  

What | const | #define
---|---|---
what it does | read-only var | substitute text
type control | yes | no
scope | local, like normal vars | global  
debugging | normal var, has name and address | invisible, it's just a number  